<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/llama_cpp/llama_cpp_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# llama.cpp on a Colab T4

Build [llama.cpp](https://github.com/ggml-org/llama.cpp) with CUDA, download a GGUF model, and chat with it.

Runtime → Change runtime type → **T4 GPU**. The first build takes several minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Config

In [ ]:
# Qwen3.8-27B, Unsloth dynamic Q4_K_M (~17.1 GB). Single-file quant.
REPO_ID   = "unsloth/Qwen3.8-27B-GGUF"
FILENAME  = "Qwen3.8-27B-UD-Q4_K_M.gguf"
# Smaller alternatives if this is too slow or runs out of memory:
#   FILENAME = "Qwen3.8-27B-UD-IQ4_XS.gguf"      (~15.7 GB, almost fits the T4 entirely)
#   REPO_ID = "bartowski/Qwen3.8-27B-GGUF"; FILENAME = "Qwen3.8-27B-Q4_K_M.gguf"

# The Q4_K_M file (17.1 GB) is larger than the T4's 16 GB, so only part of the model goes on the GPU.
# The remaining layers run on the CPU out of system RAM (~12 GB), which is much slower.
# Raise N_GPU_LAYERS until you hit an out-of-memory error, then back off a few layers.
N_GPU_LAYERS = 48     # the model has about 64 layers; 99 would offload all and fail on a T4
CTX_SIZE     = 4096   # more context = more VRAM

## 3. Build llama.cpp with CUDA (T4 = compute capability 7.5)

In [ ]:
%%capture build_log
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF
!cmake --build build --config Release -j $(nproc) --target llama-server llama-cli

In [ ]:
import os
print(os.listdir("/content/llama.cpp/build/bin")[:20])
assert os.path.exists("/content/llama.cpp/build/bin/llama-server"), "Build failed; run `build_log.show()`"

## 4. Download the model

In [ ]:
%pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir="/content/models")
print(MODEL_PATH)
!df -h /content | tail -1

## 5. Start the server in the background

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server",
     "-m", MODEL_PATH, "-ngl", str(N_GPU_LAYERS), "-c", str(CTX_SIZE),
     "--host", "127.0.0.1", "--port", "8080"],
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

## 6. Chat (OpenAI-compatible API)

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions", json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    m = r.json()["choices"][0]["message"]
    return m.get("reasoning_content") and ("[thinking]\n" + m["reasoning_content"] + "\n[answer]\n" + m["content"]) or m["content"]

print(chat("Explain what a GGUF file is in two sentences."))

## 7. Cleanup and VRAM check

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
# server.terminate()   # stop the server when finished

## Notes
- Sizes at Q4_K_M: 3B ≈ 2 GB, 7B–8B ≈ 5 GB, 14B ≈ 9 GB. Keep the total plus KV cache under 16 GB.
- Colab disk is wiped when the session ends, so models are re-downloaded each session.
- If the build fails, print `build_log.show()` to see the error.